Seguindo nosso roadmap de machine learning\
Contexto do Problema:

Imobiliária tem dificuldades para precificar novos imóveis cadastrados;
 - Muitos clientes espalhados em diversas localidades
 - Incapaz de enviar corretores para avaliações em todos os imóveis (alto custo, viagem, diária, falta de conhecimento profundo de todas as regiões)
 - Precisa semanalmente precificar imóveis para atualizar o catálogo
 - Direcionar clientes para oportunidades certas com base no perfil dos clientes cadastrados

Tipificação do problema:
Precisamos prever um número, no caso, preço dos imóveis

Definição da Variável Dependente (Target ou Alvo)
- preco_venda

Coleta -- Essa etapa estamos pressundo que a ingestão foi feita diretamente pelo Sharepoint da Imobiliária com as 2 bases em CSV

Análise Exploratória:
- Já realizada no primeiro notebook de base_estatistica
- Faremos novamente de forma mais objetiva

Diferenciando ML e Programação Tradicional
- Programa Tradicional: Regras + Dados = Resultado

- Machine Learning:
Dados + Resultados --> Algoritimo --> Modelo --Novos Dados --> Previsão

Caso de uso:
A imobiliária tem dados históricos de clientes e imóveis cadastrados
Objetivo 1: Precificar imóveis novos
Objetivo 2: Classificar Clientes

Base Imóveis:
Características X Preço --> Algoritimo --> Modelo --> Novo Imóvel --> Preço previsto



Revisando os tipos de Modelo de Aprendizado no caso concreto:

**Supervisionado**
- Temos dados rotulados (histórico de preço conhecido)
- A máquina aprende com base nos rótulos que damos ao dado; [apartamento,casa] Já existe a resposta esperada
- 2 Formas de resolver problemas: 
    - Regressão: Prever valores numéricos (preços dos imóveis)
    - Classificação: Categoria (casa, apartamento;cliente vip, cliente comum)

Algoritimos:
- Regressão linear e Regressão logística (previsão de valor; previsão categórica) - preço do imóvel;cliente compra, não compra
- Regressão não linear Modelagem de exponenciais - previsão de desgaste de materiais
- Regressão polinomial - Prever eficiência (Produção agrícola) Pouco fertilizante menos planta. Qte ideal maxima produçao, Excesso mata a planta
- SVM Máquinas de Vetores de suporte - Encontra o limite que separa 2 classes (tumor maligno, benigno)
- Naive Bayes probabilidades com dados indpendentes - (chance de palavra grátis ser SPAM ou email normal)
- KNN Vizinho mais próximo  - Classifica um determinado ponto com base na maioria dos vizinhos
- Árvores de decisão - Fluxo binário Se A -> B - Aprovar emprestimo automático( renda maior que 3k)? Restrição CPF? , então Aprova ou Não aprova sob as condições V ou F

**Retornando a Base Imóveis**
- Já exploramos a base. Mas vamos retomar aqui
- Temoso preço
O que o algoritimo faz?
- Procura Padrões
    - Maiores imóveis custam mais (tendem)
    - Mais quartos estão vinculados ao maior valor
    - A idade influencia


In [ ]:
# Importanto bibliotecas
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns

In [ ]:
# leitura do arquivo csv
df = pd.read_csv(r'C:\Users\marciohenrique\Documents\Projetos\github\ml_stat\bases_dados\base_imoveis.csv')

In [ ]:
#df.head() #lista as colunas e 5 linhas iniciais (visualiza a base)

In [ ]:
df.shape # olhando as dimensões da base

In [ ]:
df.info()

In [ ]:
# Resumo estatisitco 
df.describe()

Qualidade dos dados

In [ ]:
nulos = pd.DataFrame({
    "Nulos": df.isnull().sum(),
    "% Nulos": df.isnull().mean() * 100
})

nulos.sort_values("% Nulos", ascending=False)

Pergunta 1: Qual variável possui mais nulos?

Pergunta 2: Qual possui menos?

Pergunta 3: Qual variável é o nosso alvo?

Pergunta 4: Podemos simplesmente preencher preco_venda com a mediana?

Pergunta 5: O tratamento de area_m2 deve ser necessariamente igual ao tratamento de localidade?

A analise dos nulos depende do que queremos analisar
se o nosso alvo é o preço, e temos valores nulos, o modelo não aprende pois não há o gabarito (a supervisão)

Logo esse valor deve ser eliminado do conjunto
Para outros valores nulos, podemos simplesmente usar a mediana para preencher os nulos 
ou ainda usar a moda(valor que mais se repete nas categoricas)

| Percentual de nulos | Interpretação didática                   |
| ------------------: | ---------------------------------------- |
|                0–2% | Baixo impacto                            |
|                2–5% | Atenção                                  |
|               5–10% | Necessita tratamento                     |
|                >10% | Investigar cuidadosamente                |
|                >30% | Variável pode precisar ser reconsiderada |


In [ ]:
#df_modelo = df.dropna(subset=["preco_venda"]).copy() # removemos os nulos para o treinamento
#df["area_m2"].fillna(df["area_m2"].median()) # para area por exemplo podemos preencher os nulos pela mediana

0       152.0
1       126.0
2       587.0
3        78.0
4       255.0
        ...  
5495    244.0
5496    138.0
5497    344.0
5498    215.0
5499    519.0
Name: area_m2, Length: 5500, dtype: float64

In [22]:
# Separando as vaiaveis dependentes das independentes
# No caso nosso alvo é a variável independente ou target (preco_venda)
features = [
    "localidade",
    "tipo",
    "area_m2",
    "quartos",
    "banheiros",
    "vagas_garagem",
    "idade_anos"
]

target = "preco_venda"

X = df[features]
y = df[target]

In [23]:
#Treinamento do Modelo Definição dos parametos
#IMPORTANTE INSTALAR O SCIKIT LEARN antes !pip install scikit-learn
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

!treinamos o modelo com 80% dos dados
os 20 % restantes é a parte do conjunto testada

**Variáveis Categóricas**
OneHotEncoder é a solução para transformar nossos dados categóricos
Então por isso separamos as features numericas das não numéricas...No nosso EDA vimos que localidade e tipo são "Object" 


**Pipeline de preparação**
Pipeline é uma sequência organizada e automática de etapas que transforma os dados "como eles chegam" em dados prontos para o modelo.

Seria essa sequencia:
Dados originais → limpeza → preenchimento → transformação → modelo → previsão

o algoritmo precisa receber os dados em uma *representação numérica.*

In [24]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline # O pipeline garante que as mesmas transformações utilizadas no treinamento sejam aplicadas aos novos imóveis posteriormente.
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_features = [
    "area_m2",
    "quartos",
    "banheiros",
    "vagas_garagem",
    "idade_anos"
]

categorical_features = [
    "localidade",
    "tipo"
]

In [25]:
#Tratamento numerico
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")) # vamos usar a mediana para tratar os nulos numéricos
    ]
)

#tratamento categorico
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")), #aqui usamos a moda para imputar o mais frequente p/ substituir nulos
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)


In [26]:
#Juntanto todo o processo
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [27]:
#Algoritimo de RL
from sklearn.linear_model import LinearRegression

modelo_linear = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

In [28]:
modelo_linear.fit(X_train, y_train)

ValueError: Input y contains NaN.

**NULOS de preco_venda não tratados**
nulos das variaveis x(variaveis independentes foram tratados pela mediana e pela moda)
nulos da variavel y (preco_venda) também precisam ser tratados...
MAS, não podemos inserir mediana para ele, pois estariamos *INVENTANDO* um gabarito, compromentendo a eficiencia do modelo
logo, y_train não pode conter NAN(nulos)

antes de x e y desta celula precisa acontecer <df_modelo = df.dropna(subset=["preco_venda"]).copy()> # removemos os nulos para o treinamento

Então executamos esse processo novamente, caso tenha havido falha
Depois, contamos os null, o rsltado deve ser zero 

In [ ]:
df_modelo = df.dropna(subset=["preco_venda"]).copy() #Dropando nulos da variavel y

In [ ]:
df_modelo["preco_venda"].isnull().sum()#confirmando o resultado de drop null

np.int64(0)

In [34]:
#Recriando features e target e a divisão Treinamento e teste
features = [
    "localidade",
    "tipo",
    "area_m2",
    "quartos",
    "banheiros",
    "vagas_garagem",
    "idade_anos"
]

target = "preco_venda"

X = df_modelo[features]
y = df_modelo[target]


#Roda novamente treinamento
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [36]:
modelo_linear.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['localidade','tipo','area_m2',...,'banheiros','vagas_garagem', 'idade_anos']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of col

INTERPRETANDO O MODELO
PIPELINE É A SEQUENCIA DE ETAPAS:
1 - PROCESSOR :
2 - LINEAR REGRESSION

Processor>>preprocesso: ColumnTransformer (Idenfica e indica ao python as colunas texto e colunas numericas) -> Cada grupo deve receber um tratamento diferente
    Num e cat---foram imputadas as features os valores
        cat 2 featrures(localidade e tipo) viraram 70 features. Por que?
        ONE-Hot-Enconding fez isso...houve uma transformação de 0 e 1 para cada localidade e cada tipo...criando rotulos para esas colunas **Não foram adicionadas novas informações**
Ao final temos 75 colunas rotuladas, ou seja com seus respectivos gabaritos (Confirmando a Modelagem Supervisionada)

Conclusão: O modelo não está recebendo as 7 colunas originais. Ele está recebendo 75 colunas transformadas.

**LinearRegression**
Preço = β0 + β1Área + β2Quartos + β3Banheiros + ...
Preço =  β0 + β1 × área +β2 × quartos + ...+ β75 × alguma representação 
Ou seja o preço foi traçado pela combinação de todas as retas possiveis para cada feature para aprender a achar a melhor reta possível para precificar


Então até aqui aprendemos como o modelo foi construido

In [38]:
y_pred_linear = modelo_linear.predict(X_test)